# Lab 07-03 · Second-Order RLC Transients

Companion to chapter [07-03](../../chapters/07-03-second-order-rlc-transients.md). Handout: [README.md](README.md).

Every circuit here is written as state equations $\dot{\mathbf{x}} = \mathbf{A}\mathbf{x} + \mathbf{g}$ (chapter §3.8) and integrated numerically; the eigenvalues of $\mathbf{A}$ are the natural frequencies. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
from pathlib import Path
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)


def simulate(A, g, x0, t):
    # Integrate dx/dt = A x + g and return the states sampled at the times t (one row per state).
    A, g = np.asarray(A, float), np.asarray(g, float)
    sol = solve_ivp(lambda _t, x: A @ x + g, (t[0], t[-1]), np.asarray(x0, float), t_eval=t,
                    method="DOP853", rtol=1e-10, atol=1e-12, max_step=(t[-1] - t[0]) / 2000)
    return sol.y


def series_A(R, L, C):
    # states (i, vC): L di/dt = u - R i - vC ; C dvC/dt = i
    return np.array([[-R / L, -1 / L], [1 / C, 0.0]])


def zeta_of(A):
    ev = np.linalg.eigvals(A)
    return -ev.sum().real / 2 / np.sqrt(abs(np.prod(ev)))


def ngspice_values(*paths):
    # Run each netlist in batch mode and collect every 'name = value' line it prints.
    vals = {}
    for path in paths:
        out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
        for line in out.splitlines():
            parts = line.split("=")
            if len(parts) >= 2:
                try:
                    vals[parts[0].strip()] = float(parts[1].split()[0])
                except (ValueError, IndexError):
                    pass
    return vals


NG = ngspice_values(*sorted(str(p) for p in Path(".").glob("*.cir"))) if HAVE_NGSPICE else {}

ngspice available: True


## Part 1 — The roots move

The tank of Example 07-03.1: 100 µH and 1 µF in series with a resistor $R$.

**🔮 Predict:** at what $R$ do the two roots meet, and where do they go as $R$ grows beyond it?

In [2]:
L, C = 100e-6, 1e-6
Rs = np.linspace(0.0, 60.0, 601)
roots = np.array([np.linalg.eigvals(series_A(R, L, C)) for R in Rs])
fig, ax = plt.subplots(figsize=(5.4, 4.2))
ax.plot(roots.real.ravel() / 1e3, roots.imag.ravel() / 1e3, ".", ms=2)
th = np.linspace(0, 2 * np.pi, 400)
ax.plot(100 * np.cos(th), 100 * np.sin(th), color="0.7", lw=0.8)
ax.set_xlabel("Re s (krad/s)"); ax.set_ylabel("Im s (krad/s)"); ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
ax.set_xlim(-450, 20)
plt.show()
R_meet = brentq(lambda R: zeta_of(series_A(R, L, C)) - 1, 1, 100)
print(f"roots meet (critical damping) at R = {R_meet:.2f} ohm, at s = {np.linalg.eigvals(series_A(R_meet, L, C))[0].real / 1e3:.1f} krad/s")
assert abs(R_meet - 20.0) < 1e-6

roots meet (critical damping) at R = 20.00 ohm, at s = -100.0 krad/s


While the roots are complex they stay on a circle of radius $\omega_0$ = 100 krad/s; $R$ moves only α. At $R = 2Z_0$ = 20 Ω they meet on the real axis and then split, one toward zero (slow) and one toward $-\infty$ (fast).

## Part 2 — Every variable of one step response

Problem 07-03.14: a 5 V step into 10 Ω, 1 mH and 1 µF, at rest.

**🔮 Predict:** which of $v_C$, $i$, $v_L$, $v_R$ jump at $t = 0$, and which one does the 02-03 overshoot formula describe?

In [3]:
V, R, L, C = 5.0, 10.0, 1e-3, 1e-6
t = np.linspace(0, 1e-3, 5001)
i, vC = simulate(series_A(R, L, C), [V / L, 0], [0, 0], t)
vR = R * i
vL = V - vR - vC
fig, axes = plt.subplots(2, 2, figsize=(8.4, 5.0), sharex=True)
for ax, y, name in zip(axes.ravel(), (vC, i * 1e3, vL, vR), ("$v_C$ (V)", "$i$ (mA)", "$v_L$ (V)", "$v_R$ (V)")):
    ax.plot(t * 1e6, y); ax.set_ylabel(name); ax.grid(True, alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("time (µs)")
plt.tight_layout(); plt.show()
z = zeta_of(series_A(R, L, C))
os_formula = np.exp(-np.pi * z / np.sqrt(1 - z * z))
print(f"zeta = {z:.4f}; formula overshoot {os_formula * 100:.1f}%, simulated vC overshoot {(vC.max() / V - 1) * 100:.1f}%")
print(f"peak current {i.max() * 1e3:.1f} mA; vL starts at {vL[0]:.2f} V and dips to {vL.min():.3f} V")
assert abs(vC.max() / V - 1 - os_formula) < 1e-4
assert abs(vL.min() + 3.181) < 2e-3
if NG:
    print("ngspice:", NG["p14i"] * 1e3, "mA and", NG["p14v"], "V")
    assert abs(NG["p14v"] + 3.181) < 5e-3

zeta = 0.1581; formula overshoot 60.5%, simulated vC overshoot 60.5%
peak current 126.1 mA; vL starts at 5.00 V and dips to -3.181 V
ngspice: 126.11720000000001 mA and -3.181112 V


All four share α = 5000 s⁻¹ and ω_d = 31.2 krad/s, but only $v_C$ (the variable with no zeros) follows the overshoot formula. $v_L$ jumps to the full 5 V and then swings negative while the current decelerates.

## Part 3 — State equations for real circuits

The LC supply filter of Example 07-03.4 and the RC ladder of Example 07-03.6.

**🔮 Predict:** does the filter ring less at 10 Ω or at 100 Ω, and by how much? Can the RC ladder overshoot at all?

In [4]:
VS, Rs, L, C = 5.0, 0.03, 4.7e-6, 22e-6
t = np.linspace(0, 1e-3, 20001)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
peaks = {}
for RL in (10.0, 100.0):
    A = np.array([[-Rs / L, -1 / L], [1 / C, -1 / (RL * C)]])
    iL, v = simulate(A, [VS / L, 0], [0, 0], t)
    peaks[RL] = v.max()
    ax.plot(t * 1e3, v, label=f"$R_L$ = {RL:g} Ω: ζ = {zeta_of(A):.4f}, peak {v.max():.3f} V")
ax.set_xlabel("time (ms)"); ax.set_ylabel("output (V)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.show()
assert abs(peaks[10.0] - 9.172) < 1e-3 and abs(peaks[100.0] - 9.480) < 1e-3

A6 = np.array([[-2000.0, 1000.0], [1000.0, -1000.0]])
t6 = np.linspace(0, 20e-3, 4001)
v1, v2 = simulate(A6, [1000.0, 0], [0, 0], t6)
print(f"RC ladder: zeta = {zeta_of(A6):.3f}, roots {np.sort(np.linalg.eigvals(A6).real)}, max v2 = {v2.max():.5f} V")
assert v2.max() <= 1.0 + 1e-9
if NG:
    print("ngspice:", NG["ex4v10"], NG["ex4v100"], "V; ladder 50% at", NG["ex6load"] * 1e3, "ms")

RC ladder: zeta = 1.500, roots [-2618.03398875  -381.96601125], max v2 = 0.99944 V
ngspice: 9.171597 9.479625 V; ladder 50% at 2.22492 ms


The load's share of the damping, $Z_0/(2R_L)$, is small at either load because $Z_0$ = 0.46 Ω; the 30 mΩ series resistance does most of the work. The RC ladder's roots are real (ζ = 1.5): it creeps to 1 V and never overshoots, as §3.9 proves for every passive RC circuit.

## Part 4 — Designing the damping

**🔮 Predict:** (a) the electrolytic ESR that minimizes the hot-plug peak of Problem 07-03.11; (b) the snubber capacitance beyond which Example 07-03.5 stops improving.

In [5]:
V, L, R, C, Cb = 12.0, 1e-6, 0.05, 10e-6, 47e-6
t = np.linspace(0, 150e-6, 6001)


def hot_plug_peak(Rb):
    A = [[-R / L, -1 / L, 0], [1 / C, -1 / (Rb * C), 1 / (Rb * C)], [0, 1 / (Rb * Cb), -1 / (Rb * Cb)]]
    return simulate(A, [V / L, 0, 0], [0, 0, 0], t)[1].max()


Rbs = np.geomspace(0.05, 3.0, 40)
pk = np.array([hot_plug_peak(Rb) for Rb in Rbs])
best = Rbs[pk.argmin()]
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.3))
axes[0].semilogx(Rbs, pk, ".-"); axes[0].axvline(np.sqrt(L / C), color="0.6", ls=":")
axes[0].set_xlabel("electrolytic ESR $R_b$ (Ω)"); axes[0].set_ylabel("peak input (V)"); axes[0].grid(True, which="both", alpha=0.3)
print(f"best ESR about {best:.2f} ohm (peak {pk.min():.2f} V); Z0 of the ceramic tank = {np.sqrt(L / C):.3f} ohm")
assert 0.2 < best < 0.3

V5, L5, C5, R5 = 48.0, 2e-9, 250e-12, 0.1
Z0 = np.sqrt(L5 / C5)
t5 = np.linspace(0, 80e-9, 16001)
ks = np.array([0.5, 1, 1.5, 2, 3, 4, 6, 8, 10, 12])
peak5, settle5 = [], []
for k in ks:
    Cd = k * C5
    A = [[-R5 / L5, -1 / L5, 0], [1 / C5, -1 / (Z0 * C5), 1 / (Z0 * C5)], [0, 1 / (Z0 * Cd), -1 / (Z0 * Cd)]]
    vc = simulate(A, [V5 / L5, 0, 0], [0, 0, 0], t5)[1]
    peak5.append(vc.max())
    settle5.append(t5[np.where(np.abs(vc - V5) > 0.05 * V5)[0][-1]] * 1e9)
ax = axes[1]
ax.plot(ks, settle5, "o-", label="settling to ±5% (ns)")
ax.plot(ks, ks * C5 * V5**2 * 500e3 * 10, "s--", label="snubber power ×10 (W)")
ax.set_xlabel("$C_d / C$"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
for k, p, s_ in zip(ks, peak5, settle5):
    print(f"Cd = {k:4.1f} C: peak {p:6.2f} V, settles in {s_:5.1f} ns, snubber power {k * C5 * V5**2 * 500e3:.3f} W")
assert abs(peak5[list(ks).index(4)] - 63.67) < 0.01
if NG:
    print("ngspice peaks:", NG["p11c"], "V (hot plug, 0.3 ohm);", NG["ex5v4"], "V (snubber 4C)")

best ESR about 0.24 ohm (peak 13.59 V); Z0 of the ceramic tank = 0.316 ohm


Cd =  0.5 C: peak  86.16 V, settles in  29.7 ns, snubber power 0.144 W
Cd =  1.0 C: peak  79.06 V, settles in  15.2 ns, snubber power 0.288 W
Cd =  1.5 C: peak  74.19 V, settles in  10.2 ns, snubber power 0.432 W
Cd =  2.0 C: peak  70.76 V, settles in   7.6 ns, snubber power 0.576 W
Cd =  3.0 C: peak  66.35 V, settles in   5.0 ns, snubber power 0.864 W
Cd =  4.0 C: peak  63.67 V, settles in   5.1 ns, snubber power 1.152 W
Cd =  6.0 C: peak  60.63 V, settles in   5.0 ns, snubber power 1.728 W
Cd =  8.0 C: peak  58.98 V, settles in   4.7 ns, snubber power 2.304 W
Cd = 10.0 C: peak  57.95 V, settles in   4.5 ns, snubber power 2.880 W
Cd = 12.0 C: peak  57.25 V, settles in   4.3 ns, snubber power 3.456 W
ngspice peaks: 13.69751 V (hot plug, 0.3 ohm); 63.66789 V (snubber 4C)


The best ESR sits near $Z_0$, and the snubber's settling time stops improving beyond about $C_d$ = 3–4 $C$ while its power keeps rising in proportion: the rule of thumb of Todd's 1993 snubber paper, rediscovered numerically.

## Guided exercise — the doubled cable

Chapter §5: double the hot-plug cable (2 µH, 0.1 Ω) with the bare 10 µF ceramic. Predict $Z_0$, ζ, the peak voltage and the peak current first.

In [6]:
L2, R2 = 2e-6, 0.1
t = np.linspace(0, 200e-6, 8001)
i2, v2 = simulate(series_A(R2, L2, 10e-6), [12 / L2, 0], [0, 0], t)
print(f"Z0 = {np.sqrt(L2 / 10e-6):.4f} ohm, zeta = {zeta_of(series_A(R2, L2, 10e-6)):.4f}, "
      f"peak {v2.max():.2f} V, peak current {i2.max():.2f} A")

Z0 = 0.4472 ohm, zeta = 0.1118, peak 20.43 V, peak current 22.77 A


## Your turn

1. In Example 07-03.2, change $R_2$ from 25 Ω to 50 Ω. Classify the response and find the most negative voltage.
2. In Problem 07-03.12 (8 nH, 2 nF, 1 Ω internal), what external gate resistor gives ζ = 0.7?
3. Replace the 10 µF hot-plug ceramic with 22 µF (same 1 µH, 50 mΩ). Find $Z_0$, ζ and the peak voltage.

<details><summary>Answers</summary>

1. ζ = $Z_0/(2R_2)$ = 100/100 = 1: critically damped, with α = $1/(2R_2C)$ = 10⁴ s⁻¹. From $v(0^+) = 0$ and $\dot v(0^+) = -10^6$ V/s, $v = -10^6\,t\,e^{-10^4t}$ V, which bottoms out at $t = 1/\alpha$ = 100 µs at **−36.79 V**. A larger $R_2$ damps less and lets the kick grow toward $-I_0R_2 = -50$ V.
2. ζ = $R/(2Z_0)$ with $Z_0$ = 2 Ω, so the loop needs $R = 2\times0.7\times2 = 2.8$ Ω in total: add **1.8 Ω**.
3. $Z_0 = \sqrt{1\ \mu\text{H}/22\ \mu\text{F}}$ = **0.2132 Ω**, ζ = 0.05/0.4264 = **0.1173**, and the peak is $12\,(1 + e^{-\pi\zeta/\sqrt{1 - \zeta^2}})$ = **20.28 V**. More ceramic capacitance alone barely helps: it lowers $Z_0$, which raises ζ only slowly.

</details>

In [7]:
# Check the 'Your turn' answers by simulation.
A = np.array([[-1 / (50 * 1e-6), -1 / 1e-6], [1 / 10e-3, 0.0]])   # states (v, iL), Example 2 with R2 = 50 ohm
t = np.linspace(0, 1e-3, 20001)
v = simulate(A, [0, 0], [0, 1.0], t)[0]
print(f"1) zeta = {zeta_of(A):.3f}, minimum {v.min():.2f} V at {t[v.argmin()] * 1e6:.0f} us")
assert abs(v.min() + 36.79) < 0.01
R_ext = 2 * 0.7 * np.sqrt(8e-9 / 2e-9) - 1.0
print(f"2) external gate resistor {R_ext:.2f} ohm; check zeta = {zeta_of(series_A(1.0 + R_ext, 8e-9, 2e-9)):.3f}")
assert abs(zeta_of(series_A(1.0 + R_ext, 8e-9, 2e-9)) - 0.7) < 1e-9
t = np.linspace(0, 100e-6, 20001)
v3 = simulate(series_A(0.05, 1e-6, 22e-6), [12 / 1e-6, 0], [0, 0], t)[1]
print(f"3) Z0 = {np.sqrt(1e-6 / 22e-6):.4f} ohm, zeta = {zeta_of(series_A(0.05, 1e-6, 22e-6)):.4f}, peak {v3.max():.2f} V")
assert abs(v3.max() - 20.28) < 0.01

1) zeta = 1.000, minimum -36.79 V at 100 us
2) external gate resistor 1.80 ohm; check zeta = 0.700


3) Z0 = 0.2132 ohm, zeta = 0.1173, peak 20.28 V
